# flashattention-cuda — Colab bootstrap (T4)

One pass: confirm the GPU → build the kernel → predict the roofline → test vs SDPA → benchmark.

**Runtime → Change runtime type → T4 GPU** before running. Everything below runs on the GPU;
nothing here works on a CPU-only runtime.

## 0. Confirm the hardware (Step 0 of the brief)
We record GPU model, compute capability, and clocks — every benchmark row must carry these,
and the free-tier T4 thermally throttles.

In [20]:
!nvidia-smi --query-gpu=name,compute_cap,clocks.sm,clocks.max.sm,memory.total,temperature.gpu --format=csv
import torch
print('torch', torch.__version__, '| cuda', torch.version.cuda, '| capability', torch.cuda.get_device_capability())

name, compute_cap, clocks.current.sm [MHz], clocks.max.sm [MHz], memory.total [MiB], temperature.gpu
Tesla T4, 7.5, 300 MHz, 1590 MHz, 15360 MiB, 39
torch 2.11.0+cu128 | cuda 12.8 | capability (7, 5)


## 1. Get the repo
`REPO_URL` is already set to the public repo, so the clone just works. (If you fork it, point
`REPO_URL` at your fork.) The repo root is added to `sys.path` so the notebook can import it.

In [21]:
REPO_URL = 'https://github.com/gkienpham-cmd/flashattention-cuda.git'  # public; plain clone works on Colab
import os, sys, subprocess
if not os.path.isdir('flashattention-cuda'):
    subprocess.run(['git', 'clone', REPO_URL], check=True)
os.chdir('flashattention-cuda')
sys.path.insert(0, os.getcwd())
print('cwd', os.getcwd())

cwd /content/flashattention-cuda/flashattention-cuda


## 2. Predict the roofline BEFORE running anything
The per-step loop starts here: predict the limiter, then check it against reality below.

In [22]:
!python -m roofline.predict --arch sm_75 --shape 1x8x2048x64 --precision fp32 --materialize-s
print()
!python -m roofline.predict --arch sm_75 --shape 1x8x2048x128 --precision fp32 --materialize-s

arch        : Tesla T4 (sm_75)
shape       : B=1 H=8 N=2048 d=64  precision=fp32  materialize_S=True  tile=1x1
LIMITER     : HBM   (predicted lower bound 109.065 ms)
  t_mma     :    1.060 ms   util   1.0%
  t_hbm     :  109.065 ms   util 100.0%
  t_mufu    :    0.033 ms   util   0.0%
intensity   : 0.2 FLOP/byte   (arch ridge 25.3; BELOW -> memory-bound)

arch        : Tesla T4 (sm_75)
shape       : B=1 H=8 N=2048 d=128  precision=fp32  materialize_S=True  tile=1x1
LIMITER     : HBM   (predicted lower bound 216.452 ms)
  t_mma     :    2.121 ms   util   1.0%
  t_hbm     :  216.452 ms   util 100.0%
  t_mufu    :    0.033 ms   util   0.0%
intensity   : 0.2 FLOP/byte   (arch ridge 25.3; BELOW -> memory-bound)


## 3. Build the v1 kernel (JIT)
First call compiles with nvcc (~1 min); cached afterwards. `verbose=True` prints the build.

In [23]:
# Clean build: a failed compile (e.g. ninja missing on the first try) leaves a stale cache dir
# with a version stamp but no .so, so torch SKIPS the rebuild and then fails to import. Remove
# any partial fa_* build dir (one with no compiled .so); a good cached build is kept, so re-runs
# stay fast.
import glob, os, shutil
for d in glob.glob(os.path.expanduser('~/.cache/torch_extensions/*/fa_*')):
    if not glob.glob(os.path.join(d, '*.so')):
        shutil.rmtree(d, ignore_errors=True)
        print('cleaned stale build:', d)
print('clean-build check done')

clean-build check done


In [24]:
# cpp_extension.load() compiles via ninja, which Colab doesn't always ship — install it first.
!pip install -q ninja
from bindings.load import build_kernel
mod = build_kernel('v1_naive')
print('built:', mod)

built: <module 'fa_v1_naive' from '/root/.cache/torch_extensions/py312_cu128/fa_v1_naive/fa_v1_naive.so'>


## 4. Correctness vs SDPA (documented tolerance: atol/rtol 1e-4)

In [25]:
!python -m pytest tests/ -q

..........................                                               [100%]
26 passed in 173.75s (0:02:53)


## 5. Benchmark vs SDPA across the sweep
Expect to be **slower than SDPA** — SDPA is already a fused efficient kernel. This is the
'before'. Paste the numbers into `docs/results.md` and compare to the roofline prediction.

In [26]:
!python -m bench.harness --backend v1_naive --precision fp32

# device: Tesla T4 (sm_75)  clock~360/1590MHz  backend=v1_naive  precision=fp32  causal=False
#            shape |    ours p50/p99 ms |    sdpa p50/p99 ms |  speedup |  tok/s(ours) | roofline
ninja: no work to do.
        1x8x512x64 |   6.353/ 14.505 |   0.219/  2.255 |    0.03x |    6.447e+05 | HBM (~6.82ms)
       1x8x512x128 |  10.283/ 10.462 |   0.368/  2.420 |    0.04x |    3.983e+05 | HBM (~13.53ms)
       1x8x2048x64 |  89.029/ 92.430 |   3.410/  3.467 |    0.04x |    1.840e+05 | HBM (~109.07ms)
      1x8x2048x128 | 172.502/177.775 |   5.969/  6.522 |    0.03x |    9.498e+04 | HBM (~216.45ms)
       1x8x8192x64 | 1622.304/1644.585 |  52.938/ 53.264 |    0.03x |    4.040e+04 | HBM (~1744.88ms)
      1x8x8192x128 | 3121.138/3138.883 | 105.206/107.430 |    0.03x |    2.100e+04 | HBM (~3462.92ms)


## 6. (Optional) Nsight Compute capture
Confirms the limiter: DRAM throughput near peak at d=64 (the bandwidth wall), low MMA/MUFU.
`ncu` may need a GPU runtime that allows profiling; see profiling/GUIDE.md.

In [27]:
!bash profiling/capture.sh v1_naive || echo 'ncu unavailable on this runtime; read GUIDE.md'

==PROF== Connected to process 24568 (/usr/bin/python3.12)
# device: Tesla T4 (sm_75)  clock~300/1590MHz  backend=v1_naive  precision=fp32  causal=False
#            shape |    ours p50/p99 ms |    sdpa p50/p99 ms |  speedup |  tok/s(ours) | roofline
ninja: no work to do.
==PROF== Profiling "<unnamed>::qk_kernel(const float *, const float *, float *, int, int, int, float, long)": 0%....50%....100% - 31 passes
==PROF== Profiling "<unnamed>::softmax_kernel(float *, int, int, bool, long)": 0%....50%....100% - 31 passes
==PROF== Profiling "<unnamed>::pv_kernel(const float *, const float *, float *, int, int, int, long)": 0%....50%....100% - 31 passes
        1x8x512x64 |   6.057/  9.909 |   0.339/  1.935 |    0.06x |    6.762e+05 | HBM (~6.82ms)
       1x8x512x128 |  11.026/ 11.395 |   0.475/  0.537 |    0.04x |    3.715e+05 | HBM (~13.53ms)
       1x8x2048x64 |  95.635/ 98.794 |   3.529/  3.766 |    0.04x |    1.713e+05 | HBM (~109.07ms)
      1x8x2048x128 | 185.860/190.743 |   6.709/  7.1

In [28]:
!git pull origin main

From https://github.com/gkienpham-cmd/flashattention-cuda
 * branch            main       -> FETCH_HEAD
Already up to date.


In [29]:
!python -m pytest -q

..........................                                               [100%]
26 passed in 2.43s


In [30]:
!python -m bench.harness --backend v2_tiled --precision fp32

# device: Tesla T4 (sm_75)  clock~345/1590MHz  backend=v2_tiled  precision=fp32  causal=False
#            shape |    ours p50/p99 ms |    sdpa p50/p99 ms |  speedup |  tok/s(ours) | roofline
ninja: no work to do.
        1x8x512x64 |   4.424/  4.538 |   0.225/  0.255 |    0.05x |    9.258e+05 | HBM (~0.21ms)
       1x8x512x128 |   3.779/  3.869 |   0.384/  0.430 |    0.10x |    1.084e+06 | HBM (~0.53ms)
       1x8x2048x64 |  34.638/ 35.746 |   3.493/  3.599 |    0.10x |    4.730e+05 | HBM (~3.37ms)
      1x8x2048x128 |  60.458/ 61.233 |   6.409/  7.097 |    0.11x |    2.710e+05 | HBM (~8.41ms)
       1x8x8192x64 | 662.573/678.555 |  53.332/ 53.981 |    0.08x |    9.891e+04 | HBM (~53.74ms)
      1x8x8192x128 | 1058.247/1066.811 | 106.634/108.050 |    0.10x |    6.193e+04 | HBM (~134.32ms)


In [31]:
!python -m bench.harness --backend v1_naive --precision fp32

# device: Tesla T4 (sm_75)  clock~435/1590MHz  backend=v1_naive  precision=fp32  causal=False
#            shape |    ours p50/p99 ms |    sdpa p50/p99 ms |  speedup |  tok/s(ours) | roofline
ninja: no work to do.
        1x8x512x64 |   5.981/  9.357 |   0.207/  0.240 |    0.03x |    6.848e+05 | HBM (~6.82ms)
       1x8x512x128 |  11.335/ 11.545 |   0.388/  0.410 |    0.03x |    3.613e+05 | HBM (~13.53ms)
       1x8x2048x64 | 100.750/101.540 |   3.588/  3.941 |    0.04x |    1.626e+05 | HBM (~109.07ms)
      1x8x2048x128 | 185.751/190.001 |   6.600/  7.391 |    0.04x |    8.820e+04 | HBM (~216.45ms)
       1x8x8192x64 | 1624.952/1637.300 |  52.861/ 53.283 |    0.03x |    4.033e+04 | HBM (~1744.88ms)
      1x8x8192x128 | 3127.948/3138.455 | 104.409/105.695 |    0.03x |    2.095e+04 | HBM (~3462.92ms)


In [32]:
!bash profiling/capture.sh v2_tiled || echo 'ncu unavailable on this runtime'

==PROF== Connected to process 27965 (/usr/bin/python3.12)
# device: Tesla T4 (sm_75)  clock~345/1590MHz  backend=v2_tiled  precision=fp32  causal=False
#            shape |    ours p50/p99 ms |    sdpa p50/p99 ms |  speedup |  tok/s(ours) | roofline
ninja: no work to do.
==PROF== Profiling "void <unnamed>::qk_tiled_kernel<(int)64, (int)64, (int)64>(const float *, const float *, float *, int, int, float)": 0%....50%....100% - 31 passes
==PROF== Profiling "<unnamed>::softmax_kernel(float *, int, int, bool, long)": 0%....50%....100% - 31 passes
==PROF== Profiling "void <unnamed>::pv_tiled_kernel<(int)64, (int)64, (int)64>(const float *, const float *, float *, int, int)": 0%....50%....100% - 31 passes
        1x8x512x64 |   4.362/  4.428 |   0.311/  0.340 |    0.07x |    9.389e+05 | HBM (~0.21ms)
       1x8x512x128 |   3.804/  3.998 |   0.492/  0.516 |    0.13x |    1.077e+06 | HBM (~0.53ms)
       1x8x2048x64 |  34.226/ 35.860 |   3.508/  3.633 |    0.10x |    4.787e+05 | HBM (~3.37ms)
 

In [33]:
!git pull origin main
!pip install -q ninja
!which ncu && ncu --version

From https://github.com/gkienpham-cmd/flashattention-cuda
 * branch            main       -> FETCH_HEAD
Already up to date.
/usr/local/cuda/bin/ncu
NVIDIA (R) Nsight Compute Command Line Profiler
Copyright (c) 2018-2025 NVIDIA Corporation
Version 2025.1.1.0 (build 35528883) (public-release)


In [34]:
!bash profiling/capture.sh v1_naive

==ERROR== File v1_naive.ncu-rep already exists. Use '-f' for overwriting the file.


In [35]:
!bash profiling/capture.sh v2_tiled

==ERROR== File v2_tiled.ncu-rep already exists. Use '-f' for overwriting the file.


In [36]:
!ls -la profiling/raw/
!git add -f profiling/raw/v1_naive.ncu-rep profiling/raw/v2_tiled.ncu-rep
!git -c user.email="pgkien11@gmail.com" -c user.name="Kien Pham" commit -m "Step 2 profiling: v1/v2 ncu captures"
!git push origin main

total 5092
drwxr-xr-x 2 root root    4096 Jun 18 20:05 .
drwxr-xr-x 3 root root    4096 Jun 18 19:52 ..
-rw-r--r-- 1 root root 2556169 Jun 18 19:52 v1_naive.ncu-rep
-rw-r--r-- 1 root root 2644618 Jun 18 20:05 v2_tiled.ncu-rep
[main 69ab1aa] Step 2 profiling: v1/v2 ncu captures
 2 files changed, 0 insertions(+), 0 deletions(-)
 create mode 100644 profiling/raw/v1_naive.ncu-rep
 create mode 100644 profiling/raw/v2_tiled.ncu-rep
fatal: could not read Username for 'https://github.com': No such device or address


In [37]:
!ncu -i profiling/raw/v1_naive.ncu-rep --page raw --csv \
  --metrics dram__bytes_read.sum,gpu__dram_throughput.avg.pct_of_peak_sustained_elapsed,sm__throughput.avg.pct_of_peak_sustained_elapsed,sm__warps_active.avg.pct_of_peak_sustained_active

"ID","Process ID","Process Name","Host Name","Kernel Name","Context","Stream","Block Size","Grid Size","Device","CC","dram__bytes_read.sum","gpu__dram_throughput.avg.pct_of_peak_sustained_elapsed","sm__throughput.avg.pct_of_peak_sustained_elapsed","sm__warps_active.avg.pct_of_peak_sustained_active"
"","","","","","","","","","","","Mbyte","%","%","%"
"0","24568","python3.12","127.0.0.1","<unnamed>::qk_kernel(const float *, const float *, float *, int, int, int, float, long)","1","7","(256, 1, 1)","(8192, 1, 1)","0","7.5","3.846368","0.347459","6.122782","98.206601"
"1","24568","python3.12","127.0.0.1","<unnamed>::softmax_kernel(float *, int, int, bool, long)","1","7","(256, 1, 1)","(16, 1, 1)","0","7.5","49.866560","14.739410","2.981402","24.817685"
"2","24568","python3.12","127.0.0.1","<unnamed>::pv_kernel(const float *, const float *, float *, int, int, int, long)","1","7","(256, 1, 1)","(1024, 1, 1)","0","7.5","18.398400","7.242149","80.420141","94.581878"


In [38]:
!ncu -i profiling/raw/v2_tiled.ncu-rep --page raw --csv \
  --metrics dram__bytes_read.sum,gpu__dram_throughput.avg.pct_of_peak_sustained_elapsed,sm__throughput.avg.pct_of_peak_sustained_elapsed,sm__warps_active.avg.pct_of_peak_sustained_active

"ID","Process ID","Process Name","Host Name","Kernel Name","Context","Stream","Block Size","Grid Size","Device","CC","dram__bytes_read.sum","gpu__dram_throughput.avg.pct_of_peak_sustained_elapsed","sm__throughput.avg.pct_of_peak_sustained_elapsed","sm__warps_active.avg.pct_of_peak_sustained_active"
"","","","","","","","","","","","Mbyte","%","%","%"
"0","27965","python3.12","127.0.0.1","void <unnamed>::qk_tiled_kernel<64, 64, 64>(const float *, const float *, float *, int, int, float)","1","7","(256, 1, 1)","(8, 8, 8)","0","7.5","3.286944","1.253740","6.608012","47.647815"
"1","27965","python3.12","127.0.0.1","<unnamed>::softmax_kernel(float *, int, int, bool, long)","1","7","(256, 1, 1)","(16, 1, 1)","0","7.5","49.830912","14.501567","3.045914","24.924901"
"2","27965","python3.12","127.0.0.1","void <unnamed>::pv_tiled_kernel<64, 64, 64>(const float *, const float *, float *, int, int)","1","7","(256, 1, 1)","(8, 8, 1)","0","7.5","13.500320","6.220122","63.056177","41.985511"


In [39]:
!mkdir -p profiling/raw
!ncu --launch-count 3 --kernel-name-base demangled --kernel-name 'regex:(qk|softmax|pv)' \
  --metrics dram__bytes_read.sum,gpu__dram_throughput.avg.pct_of_peak_sustained_elapsed,sm__throughput.avg.pct_of_peak_sustained_elapsed,sm__warps_active.avg.pct_of_peak_sustained_active \
  -f -o profiling/raw/v1_n8192_d64 \
  python -c "import bench.harness as h; h.SEQ_LENS=[8192]; h.HEAD_DIMS=[64]; h.run('v1_naive','fp32',1,8,False)"

==PROF== Connected to process 28920 (/usr/bin/python3.12)
# device: Tesla T4 (sm_75)  clock~300/1590MHz  backend=v1_naive  precision=fp32  causal=False
#            shape |    ours p50/p99 ms |    sdpa p50/p99 ms |  speedup |  tok/s(ours) | roofline
ninja: no work to do.
==PROF== Profiling "<unnamed>::qk_kernel(const float *, const float *, float *, int, int, int, float, long)": 0%..
==WARNING== Launching the workload is taking more time than expected. If this continues to hang, terminate the profile and re-try by profiling the range of all related launches using '--replay-mode range'. See https://docs.nvidia.com/nsight-compute/ProfilingGuide/index.html#replay for more details.
..50%....100% - 6 passes
==PROF== Profiling "<unnamed>::softmax_kernel(float *, int, int, bool, long)": 0%....50%....100% - 6 passes
==PROF== Profiling "<unnamed>::pv_kernel(const float *, const float *, float *, int, int, int, long)": 0%....50%....100% - 6 passes
       1x8x8192x64 | 1626.349/1694.950 |  52.806

In [40]:
!ncu --launch-count 3 --kernel-name-base demangled --kernel-name 'regex:(qk|softmax|pv)' \
  --metrics dram__bytes_read.sum,gpu__dram_throughput.avg.pct_of_peak_sustained_elapsed,sm__throughput.avg.pct_of_peak_sustained_elapsed,sm__warps_active.avg.pct_of_peak_sustained_active \
  -f -o profiling/raw/v2_n8192_d64 \
  python -c "import bench.harness as h; h.SEQ_LENS=[8192]; h.HEAD_DIMS=[64]; h.run('v2_tiled','fp32',1,8,False)"


==PROF== Connected to process 29506 (/usr/bin/python3.12)
# device: Tesla T4 (sm_75)  clock~345/1590MHz  backend=v2_tiled  precision=fp32  causal=False
#            shape |    ours p50/p99 ms |    sdpa p50/p99 ms |  speedup |  tok/s(ours) | roofline
ninja: no work to do.
==PROF== Profiling "void <unnamed>::qk_tiled_kernel<(int)64, (int)64, (int)64>(const float *, const float *, float *, int, int, float)": 0%....50%....100% - 6 passes
==PROF== Profiling "<unnamed>::softmax_kernel(float *, int, int, bool, long)": 0%....50%....100% - 6 passes
==PROF== Profiling "void <unnamed>::pv_tiled_kernel<(int)64, (int)64, (int)64>(const float *, const float *, float *, int, int)": 0%....50%....100% - 6 passes
       1x8x8192x64 | 661.703/788.253 |  53.471/ 54.166 |    0.08x |    9.904e+04 | HBM (~53.74ms)
==PROF== Disconnected from process 29506
==PROF== Report: /content/flashattention-cuda/flashattention-cuda/profiling/raw/v2_n8192_d64.ncu-rep


In [41]:
!ncu -i profiling/raw/v1_n8192_d64.ncu-rep --page raw --csv --metrics dram__bytes_read.sum,gpu__dram_throughput.avg.pct_of_peak_sustained_elapsed,sm__throughput.avg.pct_of_peak_sustained_elapsed,sm__warps_active.avg.pct_of_peak_sustained_active
!ncu -i profiling/raw/v2_n8192_d64.ncu-rep --page raw --csv --metrics dram__bytes_read.sum,gpu__dram_throughput.avg.pct_of_peak_sustained_elapsed,sm__throughput.avg.pct_of_peak_sustained_elapsed,sm__warps_active.avg.pct_of_peak_sustained_active


"ID","Process ID","Process Name","Host Name","Kernel Name","Context","Stream","Block Size","Grid Size","Device","CC","dram__bytes_read.sum","gpu__dram_throughput.avg.pct_of_peak_sustained_elapsed","sm__throughput.avg.pct_of_peak_sustained_elapsed","sm__warps_active.avg.pct_of_peak_sustained_active"
"","","","","","","","","","","","Gbyte","%","%","%"
"0","28920","python3.12","127.0.0.1","<unnamed>::qk_kernel(const float *, const float *, float *, int, int, int, float, long)","1","7","(256, 1, 1)","(2097152, 1, 1)","0","7.5","0.073895","0.282967","6.127499","96.921460"
"1","28920","python3.12","127.0.0.1","<unnamed>::softmax_kernel(float *, int, int, bool, long)","1","7","(256, 1, 1)","(256, 1, 1)","0","7.5","26.333301","35.472256","3.958480","79.426469"
"2","28920","python3.12","127.0.0.1","<unnamed>::pv_kernel(const float *, const float *, float *, int, int, int, long)","1","7","(256, 1, 1)","(16384, 1, 1)","0","7.5","4.389667","6.150563","81.821035","98.969557"
"ID","Process ID","Pro